In [0]:
-- How are repeated prescriptions structured ?
WITH chronic_meds AS (
  SELECT patient_id, medication_desc, start_date, stop_ts, dispenses, duration_days
  FROM health_lakehouse.gold.fact_medications
  WHERE LOWER(medication_desc) RLIKE 'lisinopril|amlodipine|hydrochlorothiazide|losartan|metoprolol'
)
SELECT patient_id, medication_desc, start_date, stop_ts, dispenses, duration_days
FROM chronic_meds
WHERE patient_id = (SELECT patient_id FROM chronic_meds ORDER BY patient_id LIMIT 1)
ORDER BY medication_desc, start_date;

In [0]:
-- ============================================================
-- MEDICATION PERSISTENCE: 
-- how long do patients persist on chronic drug CLASSES?
--  and how often do they switch?
-- ============================================================
WITH chronic_fills AS (
  SELECT
    patient_id,
    -- normalize to drug class (strip dose/formulation)
    CASE
      WHEN LOWER(medication_desc) RLIKE 'lisinopril|enalapril|benazepril' THEN 'ACE inhibitor'
      WHEN LOWER(medication_desc) RLIKE 'losartan|valsartan|olmesartan' THEN 'ARB'
      WHEN LOWER(medication_desc) RLIKE 'amlodipine' THEN 'Calcium channel blocker'
      WHEN LOWER(medication_desc) RLIKE 'hydrochlorothiazide|furosemide' THEN 'Diuretic'
      WHEN LOWER(medication_desc) RLIKE 'metoprolol|atenolol|bisoprolol' THEN 'Beta blocker'
      WHEN LOWER(medication_desc) RLIKE 'metformin|glipizide' THEN 'Oral antidiabetic'
      WHEN LOWER(medication_desc) RLIKE 'insulin' THEN 'Insulin'
      WHEN LOWER(medication_desc) RLIKE 'statin|simvastatin|atorvastatin' THEN 'Statin'
      ELSE 'Other'
    END AS drug_class,
    LEAST(duration_days, 730) AS covered_days
  FROM health_lakehouse.gold.fact_medications
  WHERE duration_days IS NOT NULL AND duration_days > 0
),
persistence AS (
  SELECT
    patient_id,
    drug_class,
    COUNT(*) AS n_fills,
    ROUND(SUM(covered_days) / 365.25, 1) AS years_covered
  FROM chronic_fills
  WHERE drug_class <> 'Other'
  GROUP BY patient_id, drug_class
  HAVING COUNT(*) >= 3
)
SELECT
  drug_class,
  COUNT(DISTINCT patient_id) AS patients,
  ROUND(AVG(n_fills), 1) AS avg_fills,
  ROUND(AVG(years_covered), 1) AS avg_years_covered,
  ROUND(MAX(years_covered), 1) AS max_years_covered
FROM persistence
GROUP BY drug_class
ORDER BY patients DESC;

In [0]:
-- ============================================================
-- MEDICATION SWITCHING — within-class-to-class transitions
-- Detects when a patient moves from one antihypertensive class to
-- another. LEAD over each patient's class-changes in date order.
-- e.g. ACE inhibitor-> ARB is a common switch (ACE cough).
-- ============================================================
WITH classed_fills AS (
  SELECT
    patient_id,
    CASE
      WHEN LOWER(medication_desc) RLIKE 'lisinopril|enalapril|benazepril' THEN 'ACE inhibitor'
      WHEN LOWER(medication_desc) RLIKE 'losartan|valsartan|olmesartan' THEN 'ARB'
      WHEN LOWER(medication_desc) RLIKE 'amlodipine' THEN 'Calcium channel blocker'
      WHEN LOWER(medication_desc) RLIKE 'hydrochlorothiazide|furosemide' THEN 'Diuretic'
      WHEN LOWER(medication_desc) RLIKE 'metoprolol|atenolol|bisoprolol' THEN 'Beta blocker'
      ELSE 'Other'
    END AS drug_class,
    start_date
  FROM health_lakehouse.gold.fact_medications
  WHERE duration_days IS NOT NULL AND duration_days > 0
),
-- collapse consecutive same-class fills to first appearance of each class-run
class_runs AS (
  SELECT patient_id, drug_class, MIN(start_date) AS class_start
  FROM classed_fills
  WHERE drug_class <> 'Other'
  GROUP BY patient_id, drug_class
),
transitions AS (
  SELECT
    patient_id,
    drug_class AS from_class,
    LEAD(drug_class) OVER (PARTITION BY patient_id ORDER BY class_start) AS to_class
  FROM class_runs
)
SELECT
  from_class,
  to_class,
  COUNT(*) AS patients
FROM transitions
WHERE to_class IS NOT NULL
  AND from_class <> to_class
GROUP BY from_class, to_class
ORDER BY patients DESC
LIMIT 15;